In [ ]:
#  Step 1: Load & Prepare Data
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# Correct  path
df = pd.read_csv("/content/drive/MyDrive/multilabel-classification/dataset/train.csv")

label_cols = [
    'Computer Science',
    'Physics',
    'Mathematics',
    'Statistics',
    'Quantitative Biology',
    'Quantitative Finance'
]

# Combine TITLE + ABSTRACT
X = (df["TITLE"] + " " + df["ABSTRACT"]).tolist()

# Convert labels to tensor
y = torch.tensor(df[label_cols].values, dtype=torch.float32)

print("Data Loaded Successfully")
print("Number of samples:", len(X))
print("Label shape:", y.shape)

In [ ]:
# ================== IMPORTS ==================
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm
import numpy as np
import pandas as pd

In [ ]:
# ================== HYPERPARAMS ==================
MODEL_NAME = "roberta-base"
MAX_LEN = 512        # based on your token length analysis
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
N_FOLDS = 5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# ================== DATA ==================
# Assume df has 'text' and labels columns
# text = title + abstract
label_cols = ['Computer Science', 'Physics', 'Mathematics', 'Statistics', 'Quantitative Biology', 'Quantitative Finance']

df['text'] = df['TITLE'] + " " + df['ABSTRACT']
X = df['text'].tolist()
y = df[label_cols].values.astype(np.float32)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:

# ================== DATASET ==================
class MultiLabelDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.texts = texts
        self.labels = torch.tensor(labels, dtype=torch.float32)
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding='max_length',
            max_length=MAX_LEN,
            return_tensors='pt'
        )
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item['labels'] = self.labels[idx]
        return item



In [ ]:
# ==================  SAFETY SETTINGS ==================
import sys, os

CHECKPOINT_DIR = "/content/drive/MyDrive/multilabel-classification/working/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("Checkpoint directory created:", CHECKPOINT_DIR, flush=True)

# ================== K-FOLD TRAINING ==================
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
    print(f"\n================ FOLD {fold+1} ================", flush=True)

    train_texts, val_texts = [X[i] for i in train_idx], [X[i] for i in val_idx]
    train_labels, val_labels = y[train_idx], y[val_idx]

    train_dataset = MultiLabelDataset(train_texts, train_labels, tokenizer)
    val_dataset = MultiLabelDataset(val_texts, val_labels, tokenizer)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=y.shape[1],
        problem_type="multi_label_classification"
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
    criterion = nn.BCEWithLogitsLoss()

    for epoch in range(EPOCHS):

        # ================= TRAIN =================
        model.train()
        train_loss = 0
        train_targets = []
        train_logits = []

        for batch in tqdm(train_loader, desc=f"Fold {fold+1} Train Epoch {epoch+1}"):
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            target = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits

            loss = criterion(logits, target)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_targets.extend(target.detach().cpu().numpy())
            train_logits.extend(logits.detach().cpu().numpy())

        avg_train_loss = train_loss / len(train_loader)
        train_targets = np.array(train_targets)
        train_preds = (np.array(train_logits) > 0).astype(int)

        # ===== Train Metrics =====
        train_subset_acc = accuracy_score(train_targets, train_preds)
        train_label_acc = (train_targets == train_preds).mean()
        train_precision = precision_score(train_targets, train_preds, average='micro', zero_division=0)
        train_recall = recall_score(train_targets, train_preds, average='micro', zero_division=0)
        train_f1_micro = f1_score(train_targets, train_preds, average='micro', zero_division=0)
        train_f1_macro = f1_score(train_targets, train_preds, average='macro', zero_division=0)

        # ================= VALIDATION =================
        model.eval()
        val_loss = 0
        val_targets = []
        val_logits = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Fold {fold+1} Val Epoch {epoch+1}"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                target = batch['labels'].to(device)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits

                loss = criterion(logits, target)
                val_loss += loss.item()

                val_targets.extend(target.detach().cpu().numpy())
                val_logits.extend(logits.detach().cpu().numpy())

        avg_val_loss = val_loss / len(val_loader)
        val_targets = np.array(val_targets)
        val_preds = (np.array(val_logits) > 0).astype(int)

        # ===== Validation Metrics =====
        val_subset_acc = accuracy_score(val_targets, val_preds)
        val_label_acc = (val_targets == val_preds).mean()
        val_precision = precision_score(val_targets, val_preds, average='micro', zero_division=0)
        val_recall = recall_score(val_targets, val_preds, average='micro', zero_division=0)
        val_f1_micro = f1_score(val_targets, val_preds, average='micro', zero_division=0)
        val_f1_macro = f1_score(val_targets, val_preds, average='macro', zero_division=0)

        # ================= PRINT METRICS =================
        print(f"\nEpoch {epoch+1}/{EPOCHS} Fold {fold+1}", flush=True)
        print("="*50, flush=True)
        print("TRAIN METRICS", flush=True)
        print(f"Train Loss: {avg_train_loss:.4f}", flush=True)
        print(f"Subset Accuracy: {train_subset_acc:.4f}", flush=True)
        print(f"Label Accuracy: {train_label_acc:.4f}", flush=True)
        print(f"Precision: {train_precision:.4f}", flush=True)
        print(f"Recall: {train_recall:.4f}", flush=True)
        print(f"F1 Micro: {train_f1_micro:.4f}", flush=True)
        print(f"F1 Macro: {train_f1_macro:.4f}", flush=True)

        print("\nVALIDATION METRICS", flush=True)
        print(f"Val Loss: {avg_val_loss:.4f}", flush=True)
        print(f"Subset Accuracy: {val_subset_acc:.4f}", flush=True)
        print(f"Label Accuracy: {val_label_acc:.4f}", flush=True)
        print(f"Precision: {val_precision:.4f}", flush=True)
        print(f"Recall: {val_recall:.4f}", flush=True)
        print(f"F1 Micro: {val_f1_micro:.4f}", flush=True)
        print(f"F1 Macro: {val_f1_macro:.4f}", flush=True)

        # ================= SAVE CHECKPOINT =================
        checkpoint_path = f"{CHECKPOINT_DIR}/{MODEL_NAME}_fold{fold+1}_epoch{epoch+1}.pt"
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Checkpoint saved → {checkpoint_path}", flush=True)

    # ================= STORE FOLD RESULTS =================
    fold_results.append({
        'train_loss': avg_train_loss,
        'val_loss': avg_val_loss,
        'train_subset_acc': train_subset_acc,
        'val_subset_acc': val_subset_acc,
        'train_label_acc': train_label_acc,
        'val_label_acc': val_label_acc,
        'train_precision': train_precision,
        'val_precision': val_precision,
        'train_recall': train_recall,
        'val_recall': val_recall,
        'train_f1_micro': train_f1_micro,
        'val_f1_micro': val_f1_micro,
        'train_f1_macro': train_f1_macro,
        'val_f1_macro': val_f1_macro
    })

    # ================= SAVE RESULTS AFTER EACH FOLD =================
    results_path = "/content/drive/MyDrive/multilabel-classification/working/kfold_results.csv"
    pd.DataFrame(fold_results).to_csv(results_path, index=False)
    print(f"Fold results saved → {results_path}", flush=True)

In [ ]:
# ================== FINAL 5-FOLD AVERAGE ==================
results_df = pd.DataFrame(fold_results)
print("\n===== 5-Fold Cross Validation Results =====")
print(results_df)

print("\n===== Average ± Std Across Folds =====")
for col in results_df.columns:
    print(f"{col}: {results_df[col].mean():.4f} ± {results_df[col].std():.4f}")

In [ ]:
# ================== FINAL 5-FOLD AVERAGE ==================
results_df = pd.DataFrame(fold_results)

print("\n===== 5-Fold Cross Validation Results =====", flush=True)
print(results_df, flush=True)

print("\n===== Average ± Std Across Folds =====", flush=True)
for col in results_df.columns:
    print(f"{col}: {results_df[col].mean():.4f} ± {results_df[col].std():.4f}", flush=True)